# Module 10A Applied Lab
## Production AI Service Stack: FastAPI · Neo4j · Weaviate · Next.js · Docker

**Session anchor:** 110 minutes
**SI Demo domain:** Movie information service — movie-NER fixture + movie-review RAG corpus
**Your assignment domain:** Recipe service — Week 9B Neo4j graph + Module 8 RAG corpus

---

### Learning Objectives

By the end of this lab you will be able to:

1. Define Pydantic request/response models and wire them to a stateless NLP endpoint
2. Inject an async Neo4j driver via FastAPI's `lifespan` + `Depends()` pattern
3. Implement a full **retrieve → assemble → generate → cite** RAG pipeline
4. Write `/healthz` and `/readyz` probes with two-backend readiness logic
5. Configure `CORSMiddleware` so a browser can consume your API
6. Scaffold a Next.js page that fetches from a Python API using `useState` + `fetch`
7. Write a single-stage Python Dockerfile and a two-stage Node Dockerfile
8. Author a two-service `docker-compose.yml` and bring the full stack up

---

| Beat | Topic | Duration |
|------|-------|----------|
| 0 | Setup orientation | 5 min |
| 1 | `/extract` — stateless NLP wrap | 12 min |
| 2 | Neo4j dependency injection + lifespan | 12 min |
| 3 | RAG: retrieve → assemble → generate → cite | 25 min |
| 4 | `/healthz` and `/readyz` with backend probes | 8 min |
| 5 | CORS + browser clickability | 5 min |
| 6 | Next.js page scaffold for Python engineers | 15 min |
| 7 | Backend Dockerfile | 10 min |
| 8 | Frontend Dockerfile — multi-stage Node | 10 min |
| 9 | Lab-level Compose — `api` + `web` | 5 min |
| 10 | Pivot to assignment | 8 min |

> Time allocations sum to 115 min, within ±10 % of the 110-min session anchor.


---
## Technical Setup Checklist

> Complete this checklist **at least 30 minutes before session start**
> so the full demo stack runs cleanly.

<ul>
<li><input type="checkbox"> Docker Desktop running;
    <code>docker compose version</code> reports v2.</li>
<li><input type="checkbox"> Node.js LTS 20.x installed;
    <code>node --version</code> reports <code>v20.x</code>.</li>
<li><input type="checkbox"> Python 3.11 venv active; all packages installed:<br>
    <code>pip install fastapi "uvicorn[standard]" pydantic neo4j spacy
    weaviate-client sentence-transformers transformers requests</code></li>
<li><input type="checkbox"> spaCy English model downloaded:
    <code>python -m spacy download en_core_web_sm</code></li>
<li><input type="checkbox"> Movie Neo4j fixture pre-loaded (~30 nodes:
    <code>(:Movie)</code>, <code>(:Director)</code>, <code>(:Actor)</code>);
    container named <code>m10-demo-neo4j</code> on port
    <strong>7688:7687</strong> (avoids collision with Lab recipe Neo4j on 7687).</li>
<li><input type="checkbox"> Movie-review Weaviate fixture pre-loaded (~30 chunks,
    embedded by <code>sentence-transformers</code>);
    container named <code>m10-demo-weaviate</code> on port
    <strong>8081:8080</strong>.</li>
<li><input type="checkbox"> <code>google/flan-t5-base</code> model cached:<br>
    <code>python -c "from transformers import pipeline;
    pipeline('text2text-generation', model='google/flan-t5-base')"</code></li>
<li><input type="checkbox"> Demo movie-NER fixture text file ready
    (Beat 1 pre-identified artifact).</li>
<li><input type="checkbox"> Slide deck pages 4, 7, 8, 9, 10 cued up
    (RAG prompt template; Next.js routing diagram; layer-cache diagram;
    image-size comparison; Compose printout).</li>
<li><input type="checkbox"> Browser tabs open:
    <code>http://localhost:8001/docs</code> (Swagger UI);
    <code>http://localhost:3001</code> (demo Next.js);
    browser console panel visible.</li>
<li><input type="checkbox"> Terminal panes pre-arranged:
    one for <code>docker compose logs -f</code>;
    one for live coding;
    one for <code>curl</code> smoke tests.</li>
</ul>


In [1]:
# ── Environment Verification ────────────────────────────────────────────────────
# Run this cell first. Fix every [MISSING] before proceeding to Beat 0.

import subprocess                                                   # Execute shell commands from Python to check external tools
import importlib                                                    # Dynamically import packages to verify they are installed
import sys                                                          # Access Python version string

# Map pip install names → Python import names (they differ for several packages)
REQUIRED_PACKAGES = {
    "fastapi":               "fastapi",
    "uvicorn":               "uvicorn",
    "pydantic":              "pydantic",
    "neo4j":                 "neo4j",
    "spacy":                 "spacy",
    "weaviate-client":       "weaviate",
    "sentence-transformers": "sentence_transformers",
    "transformers":          "transformers",
    "requests":              "requests",
}

print(f"Python {sys.version.split()[0]}\n")                         # Show Python version for debugging

all_ok = True                                                       # Flag — will be set False if any dependency is missing

# Iterate and attempt each import; report pass/fail
for pip_name, import_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(import_name)                        # Attempt the import
        print(f"  [OK]      {pip_name}")
    except ImportError:
        print(f"  [MISSING] {pip_name}  ->  pip install {pip_name}")
        all_ok = False                                              # Mark failure; keep checking the rest

# Check Docker Compose v2 (v1 uses 'docker-compose', v2 uses 'docker compose')
dc = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True)
if dc.returncode == 0:
    print(f"\n  [OK]      docker compose  —  {dc.stdout.strip()}")
else:
    print("\n  [MISSING] docker compose v2  ->  install Docker Desktop")
    all_ok = False

# Check Node.js; must be 20.x LTS (Next.js 14+ requires it)
node = subprocess.run(["node", "--version"], capture_output=True, text=True)
node_ver = node.stdout.strip()
if node.returncode == 0 and node_ver.startswith("v20"):
    print(f"  [OK]      node  —  {node_ver}")
else:
    print(f"  [WARN]    node  —  {node_ver}  (expected v20.x LTS)")

# Final summary line
print()
print("All dependencies satisfied — ready for Beat 0." if all_ok else
      "Install missing packages, then re-run this cell.")


Python 3.13.14

  [OK]      fastapi
  [OK]      uvicorn
  [OK]      pydantic


/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


  [OK]      neo4j
  [OK]      spacy
  [OK]      weaviate-client


/home/brandon/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  [OK]      sentence-transformers
  [OK]      transformers
  [OK]      requests

  [OK]      docker compose  —  Docker Compose version v2.25.0
  [WARN]    node  —  v12.22.9  (expected v20.x LTS)

All dependencies satisfied — ready for Beat 0.


---
## Beat 0 · Setup Orientation  *(5 min)*

### What your Lab template contains (recipe domain)

| Artifact | Purpose |
|----------|---------|
| `mapper/` | Week 9B ingredient → Neo4j node mapper |
| `fixtures/neo4j/` | Pre-loaded recipe graph (~60 nodes, `(:Recipe)-[:USES]->(:Ingredient)`) |
| `rag_pipeline/` | Module 8 RAG pipeline wired to a recipe-review Weaviate collection |
| `frontend/` | Pre-scaffolded Next.js project (pages exist but fetch calls are empty) |
| `.env.template` | Copy to `.env.local` — contains Neo4j/Weaviate/API URL vars |

### What the SI demo shows (movie domain)

The SI live-codes against a **movie** domain so you can see every pattern without
accidentally completing the assignment. Decision points, data shapes, and wiring
are identical — only the domain vocabulary differs.

### Port map — demo stack vs. Lab stack (no collisions)

| Service | SI Demo port | Lab (your) port |
|---------|-------------|-----------------|
| FastAPI | 8001 | 8000 |
| Neo4j Bolt | 7688 | 7687 |
| Weaviate | 8081 | 8080 |
| Next.js | 3001 | 3000 |

### Architecture diagram

```
             Browser
                |
         Next.js :3000          (recipe search UI)
                |  fetch /extract  /kg/query  /rag/answer
         FastAPI :8000          (recipe AI service)
           /         \
    Neo4j :7687    Weaviate :8080
  (recipe graph)  (review embeddings)
```


In [2]:
# ── Create Demo Directory Structure ─────────────────────────────────────────────
# All files written by %%writefile land under demo/ to stay separate
# from the Lab template's recipe-domain code.

import pathlib                                                      # Object-oriented filesystem paths — safer than raw os.path strings

# Define every directory the demo needs
DEMO_DIRS = [
    "demo",                                                         # Root — holds all versioned main_vN.py files
    "demo/frontend",                                                # Next.js project root
    "demo/frontend/pages",                                          # Next.js file-based routing: one file = one URL
    "demo/frontend/styles",                                         # CSS modules (not authored in this lab, but expected)
]

# Create each directory; parents=True creates intermediate dirs, exist_ok prevents errors
for d in DEMO_DIRS:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
    print(f"  created  {d}/")

print("\nDemo directory structure ready.")


  created  demo/
  created  demo/frontend/
  created  demo/frontend/pages/
  created  demo/frontend/styles/

Demo directory structure ready.


---
## Beat 1 · Typed Endpoint #1 — Stateless NLP Wrap  *(12 min)*

### What we are building

`POST /extract` — accepts JSON text, returns a structured list of named-entity spans.

```
Client  ──POST /extract──►  FastAPI  ──nlp()──►  spaCy
        { "text": "..." }             ◄──────────  doc.ents
        ◄──────────────────  { entities: [...], entity_count: N }
```

### Why the model loads at module scope

```python
# CORRECT — loads once when the process starts
nlp = spacy.load("en_core_web_sm")

@app.post("/extract")
async def extract(req: ExtractRequest):
    doc = nlp(req.text)   # re-uses the already-loaded pipeline
    ...
```

```python
# WRONG — 1-2 s cold-start on every single request
@app.post("/extract")
async def extract(req: ExtractRequest):
    nlp = spacy.load("en_core_web_sm")   # loaded fresh every call
    ...
```

You will hit this trap again with the generator model in Beat 3.

### Pre-identified artifacts

- **spaCy model:** `en_core_web_sm` (already installed)
- **Fixture text:**
  `"Christopher Nolan directed Interstellar starring Matthew McConaughey and Anne Hathaway."`


In [3]:
%%writefile demo/main_v1.py
# ── demo/main_v1.py ──────────────────────────────────────────────────────────────
# Beat 1: stateless NLP endpoint.
# SI domain: movies.  Lab domain (your assignment): recipes.

# FastAPI — the web framework; handles routing, dependency injection, OpenAPI docs
from fastapi import FastAPI

# BaseModel — Pydantic's base class for request/response schema declaration
from pydantic import BaseModel

# List — standard generic for typed collections in response models
from typing import List

# spaCy — NLP library; provides tokenisation, POS-tagging, dependency parsing, NER
import spacy

# ── Model loading at module scope ────────────────────────────────────────────────
# Loaded once when the Python process starts, not on every request.
# en_core_web_sm: small English pipeline — tokeniser + tagger + parser + NER
nlp = spacy.load("en_core_web_sm")

# ── FastAPI application instance ─────────────────────────────────────────────────
# title and version appear in the Swagger UI header at /docs
app = FastAPI(
    title="Movie Information Service — SI Demo",                    # Human-readable name for OpenAPI
    version="1.0.0",                                                # Semantic version shown in /docs
    description="Module 10A SI demonstration (movie domain).",      # /docs description
)

# ── Request model ────────────────────────────────────────────────────────────────
class ExtractRequest(BaseModel):
    text: str                                                       # Raw text to analyse — movie synopsis, review, or plot description

# ── Response sub-model: one named-entity span ─────────────────────────────────
class Entity(BaseModel):
    text:  str                                                      # Surface form of the entity as it appears in the original text
    label: str                                                      # spaCy entity type: PERSON, WORK_OF_ART, ORG, DATE, GPE, …
    start: int                                                      # Character offset (inclusive) at which the span begins in text
    end:   int                                                      # Character offset (exclusive) at which the span ends in text

# ── Response model: top-level extraction result ───────────────────────────────
class ExtractResponse(BaseModel):
    entities:     List[Entity]                                      # All recognised named-entity spans, in order
    entity_count: int                                               # len(entities) — client convenience field

# ── Endpoint: POST /extract ───────────────────────────────────────────────────
# response_model= tells FastAPI to validate the return value before serialisation;
# FastAPI also uses it to generate the Swagger response schema automatically.
@app.post(
    "/extract",
    response_model=ExtractResponse,                                 # Validated and documented automatically
    summary="Named-entity extraction",                              # Short description in Swagger UI
    tags=["NLP"],                                                   # Groups related endpoints in Swagger
)
async def extract_entities(request: ExtractRequest) -> ExtractResponse:
    # nlp(text) runs every stage in the pipeline: tokenise → POS → dep → NER
    doc = nlp(request.text)

    # Build one typed Entity per spaCy-identified span
    entities: List[Entity] = [
        Entity(
            text=ent.text,                                          # Matched substring from the original text
            label=ent.label_,                                       # String label (trailing _ = string, not int)
            start=ent.start_char,                                   # Char-level start index (not token index)
            end=ent.end_char,                                       # Char-level end index (exclusive)
        )
        for ent in doc.ents                                         # doc.ents: tuple of Span objects tagged as entities
    ]

    # FastAPI serialises ExtractResponse to JSON; Pydantic validates every field
    return ExtractResponse(entities=entities, entity_count=len(entities))


Overwriting demo/main_v1.py


In [4]:
# ── Beat 1: Smoke Test — /extract ────────────────────────────────────────────────
# Starts the v1 server in a background subprocess, sends a test request,
# prints the response, then stops the server.

import subprocess                                                   # Spawn the uvicorn process without blocking the kernel
import time                                                         # Brief sleep so uvicorn has time to bind the port
import json                                                         # Pretty-print the JSON response
import requests                                                     # HTTP client for the smoke-test POST

DEMO_PORT = 8001                                                    # Demo API port — distinct from the Lab's port 8000

# Start uvicorn in the background; stderr goes to PIPE so startup noise is captured
server = subprocess.Popen(                                          # Non-blocking: returns immediately after spawning
    ["uvicorn", "demo.main_v1:app",                                 # Module path: demo/main_v1.py → app object
     "--port", str(DEMO_PORT),                                      # Bind to the demo port
     "--log-level", "warning"],                                     # Suppress INFO noise in the notebook output
    stderr=subprocess.PIPE,                                         # Capture stderr so it does not clutter output
)

time.sleep(2)                                                       # Allow uvicorn startup time (typically < 1 s, 2 s is conservative)

# Movie-domain fixture text — the pre-identified Beat 1 artifact
FIXTURE_TEXT = (                                                    # Parentheses allow implicit string concatenation across lines
    "Christopher Nolan directed Interstellar, starring Matthew McConaughey, "
    "Anne Hathaway, and Jessica Chastain. The film was produced by Warner Bros."
)

try:                                                                # Guard against network/server errors without crashing the kernel
    # POST to /extract with the fixture text
    resp = requests.post(                                           # Synchronous HTTP POST — blocks until response is received
        f"http://localhost:{DEMO_PORT}/extract",                    # Target the running demo server
        json={"text": FIXTURE_TEXT},                                # Pydantic will validate this body
        timeout=5,                                                  # Fail fast if the server is not up
    )
    resp.raise_for_status()                                         # Raise an exception for 4xx/5xx responses

    # Pretty-print the JSON response so the entity structure is readable
    print(json.dumps(resp.json(), indent=2))                        # indent=2 adds human-readable whitespace

except requests.exceptions.RequestException as exc:                 # Catches connection errors, timeouts, and HTTP errors
    # Graceful failure — show the error without crashing the notebook
    print(f"Request failed: {exc}")
    print("Ensure spaCy en_core_web_sm is installed: python -m spacy download en_core_web_sm")

finally:                                                            # Always executes — ensures server is stopped even on error
    server.terminate()                                              # Stop the uvicorn process regardless of test outcome
    server.wait()                                                   # Wait for the process to fully exit before continuing
    print("\nServer stopped.")                                     # Confirm cleanup so the port is free for the next beat


Request failed: HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /extract (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ab9c182ecf0>: Failed to establish a new connection: [Errno 111] Connection refused'))
Ensure spaCy en_core_web_sm is installed: python -m spacy download en_core_web_sm

Server stopped.


---
## Beat 2 · Dependency Injection + Lifespan for the KG Backend  *(12 min)*

### What we are building

- A **lifespan context manager** that opens the Neo4j driver at startup and closes it at shutdown
- A **`get_driver` dependency function** injected via `Depends()` into each route that needs Neo4j
- `POST /kg/query` — accepts a Cypher query, returns matched records as JSON

### Why lifespan instead of a global variable

```python
# WRONG — global variable means the driver is created at import time,
# before the event loop starts, and is never explicitly closed.
driver = AsyncGraphDatabase.driver(URI, auth=AUTH)

# CORRECT — lifespan guarantees startup/shutdown order
# and gives you async context in both phases.
@asynccontextmanager
async def lifespan(app):
    driver = AsyncGraphDatabase.driver(URI, auth=AUTH)
    app.state.neo4j_driver = driver   # share via app.state
    yield                              # app runs here
    await driver.close()              # guaranteed cleanup
```

### Why Depends() instead of a global session

- Each request gets its own session (no cross-request state pollution)
- FastAPI handles teardown automatically when the generator yields
- Makes the dependency explicit and testable

### Pre-identified artifact

Movie Neo4j fixture: `(:Movie)-[:DIRECTED_BY]->(:Director)` — ~30 nodes on port 7688.


In [5]:
# ── Beat 2: Smoke Test — /kg/query ──────────────────────────────────────────────
# Requires the m10-demo-neo4j container to be running on port 7688.
# If it is not, the server will start but /kg/query will return a 500.

import subprocess, time, json, requests                             # Standard-library + third-party imports for server and HTTP

DEMO_PORT = 8001                                                    # Same port — we stop/start between beats in the demo

server = subprocess.Popen(                                          # Launch v2 (Neo4j lifespan) as a background process
    ["uvicorn", "demo.main_v2:app",                                 # v2 module with Neo4j lifespan
     "--port", str(DEMO_PORT),                                      # Bind to the demo port
     "--log-level", "warning"],                                     # Suppress INFO noise in the notebook output
    stderr=subprocess.PIPE,                                         # Capture stderr to keep output clean
)
time.sleep(2)                                                       # Wait for uvicorn + lifespan startup

# Cypher query — returns all movies and their directors from the demo fixture
CYPHER = (                                                          # Multi-line string via implicit concatenation
    "MATCH (m:Movie)-[:DIRECTED_BY]->(d:Director)\n"              # Traverse the DIRECTED_BY relationship
    "RETURN m.title AS movie, d.name AS director\n"               # Project title and director name as aliases
    "ORDER BY m.title\n"                                          # Sort results alphabetically by movie title
    "LIMIT 5"                                                       # Return at most 5 rows
)

try:                                                                # Guard against connection errors if Neo4j is not running
    resp = requests.post(
        f"http://localhost:{DEMO_PORT}/kg/query",                   # Target the KG endpoint
        json={"cypher": CYPHER, "params": {}},                      # Empty params for this read query
        timeout=5,                                                   # Fail fast if the server is not up
    )
    resp.raise_for_status()                                         # Raise HTTPError for 4xx/5xx responses
    print(json.dumps(resp.json(), indent=2))                        # Print returned records

except requests.exceptions.RequestException as exc:                 # Catches network errors and HTTP errors
    print(f"Request failed (is m10-demo-neo4j running on port 7688?): {exc}")

finally:                                                            # Always runs — stops the server whether or not the test passed
    server.terminate()                                              # Send SIGTERM to uvicorn
    server.wait()                                                   # Block until the process exits
    print("\nServer stopped.")                                     # Confirm cleanup completed


/tmp/ipykernel_23804/3048487492.py:9: ResourceWarning: unclosed file <_io.BufferedReader name=77>
  server = subprocess.Popen(                                          # Launch v2 (Neo4j lifespan) as a background process


Request failed (is m10-demo-neo4j running on port 7688?): HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /kg/query (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ab9c18196d0>: Failed to establish a new connection: [Errno 111] Connection refused'))

Server stopped.


---
## Beat 3 · RAG Wrap — Retrieve → Assemble → Generate → Cite  *(25 min)*

This is the **densest beat**. The SI walks each sub-step in order.

### The four-step pipeline

```
(a) Weaviate near_text(k=4)       → top-4 relevant chunks
(b) assemble_prompt(chunks, Q)    → numbered context + question
(c) flan-t5-base(prompt)          → raw generated answer
(d) extract_citations(answer)     → list of [N] references
(e) grounding check               → no citations? return sentinel
```

### The prompt template (Slide 4)

```
You are a movie expert. Use ONLY the excerpts below to answer.
Cite each excerpt you use with its number in square brackets.

[1] <chunk 1 text>
[2] <chunk 2 text>
[3] <chunk 3 text>
[4] <chunk 4 text>

Question: {question}
Answer:
```

### The grounding refusal pattern

If the model generates text with **no `[N]` citations**, we cannot verify
which source supported the claim. We return a sentinel response instead
of potentially hallucinated text.

```python
if not citations:
    return RAGResponse(
        answer=GROUNDING_SENTINEL,   # "I cannot answer from provided sources."
        citations=[],
        grounded=False,
    )
```

### Pre-identified artifacts

- **Weaviate:** `m10-demo-weaviate` on port 8081, class `MovieReview`
- **Generator:** `google/flan-t5-base` — already in `~/.cache/huggingface`
- **Prompt template:** printed on Slide 4


In [6]:
# ── Beat 3: Smoke Test — /rag/answer ─────────────────────────────────────────────
# Requires m10-demo-weaviate on port 8081.
# flan-t5-base will load from ~/.cache/huggingface (takes ~30 s first time if not cached).

import subprocess, time, json, requests                             # Standard-library + third-party imports

DEMO_PORT = 8001                                                    # Demo port — stop/start between beats

server = subprocess.Popen(                                          # Launch v3 (Weaviate + RAG) as a background process
    ["uvicorn", "demo.main_v3:app",                                 # v3 module: adds /rag/answer endpoint
     "--port", str(DEMO_PORT),                                      # Bind to the demo port
     "--log-level", "warning"],                                     # Suppress INFO noise in the notebook output
    stderr=subprocess.PIPE,                                         # Capture stderr to avoid cluttering cell output
)

# v3 startup is slower because flan-t5-base loads — wait up to 45 s
print("Waiting for server + model load (up to 45 s) ...")          # Inform the user this step takes time
time.sleep(30)                                                      # flan-t5-base loads in ~20-30 s on first invocation

try:                                                                # Guard against network/server errors
    resp = requests.post(                                           # Synchronous HTTP POST — blocks until response
        f"http://localhost:{DEMO_PORT}/rag/answer",                 # RAG endpoint added in v3
        json={"question": "What makes Interstellar visually distinctive?"},  # Test question about the movie domain
        timeout=60,                                                 # Generation can take several seconds on CPU
    )
    resp.raise_for_status()                                         # Raise HTTPError for 4xx/5xx responses
    data = resp.json()                                              # Parse the JSON response body

    print(f"Answer: {data['answer']}")                              # Model-generated answer grounded in retrieved chunks
    print(f"Grounded: {data['grounded']}")                          # True if at least one chunk cited; False if fallback sentinel used
    print(f"Citations ({len(data['citations'])}):")                 # Show how many chunks were cited
    for c in data["citations"]:                                     # Iterate over each cited chunk
        # Show a truncated preview of each cited chunk
        preview = c["text"][:120].replace("\n", " ")              # Truncate to 120 chars and collapse newlines for readability
        print(f"  [{c['index']}] {preview}...")                     # Display citation index and chunk preview

except requests.exceptions.RequestException as exc:                 # Catches connection refused, timeout, HTTP errors
    print(f"Request failed: {exc}")                                 # Surface the error message
    print("Ensure m10-demo-weaviate is running on port 8081 and flan-t5-base is cached.")

finally:                                                            # Always runs — guaranteed cleanup
    server.terminate()                                              # Send SIGTERM to uvicorn
    server.wait()                                                   # Block until the process exits cleanly
    print("\nServer stopped.")                                     # Confirm cleanup completed


Waiting for server + model load (up to 45 s) ...


/tmp/ipykernel_23804/4266603210.py:9: ResourceWarning: unclosed file <_io.BufferedReader name=79>
  server = subprocess.Popen(                                          # Launch v3 (Weaviate + RAG) as a background process


Request failed: HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /rag/answer (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ab9c181a490>: Failed to establish a new connection: [Errno 111] Connection refused'))
Ensure m10-demo-weaviate is running on port 8081 and flan-t5-base is cached.

Server stopped.


---
## Beat 4 · `/healthz` and `/readyz` with Two Backend Probes  *(8 min)*

### Liveness vs. Readiness

| Probe | Path | Question answered | Failing action |
|-------|------|------------------|----------------|
| Liveness | `/healthz` | Is the process alive? | Restart the container |
| Readiness | `/readyz` | Can the process serve traffic? | Remove from load balancer |

### The readiness pattern

```
/readyz                                                   200 OK
  ├── probe Neo4j   →  RETURN 1         (fast Cypher ping)
  └── probe Weaviate →  GET /v1/.well-known/ready          503 if either fails
```

### Demo: stop Weaviate, observe the flip

```bash
# Terminal 1 — watch the probe in a loop
watch -n1 'curl -s http://localhost:8001/readyz | jq .status'

# Terminal 2 — stop Weaviate
docker stop m10-demo-weaviate

# /readyz flips to 503 within 1 s; restart Weaviate to observe recovery
docker start m10-demo-weaviate
```

### Pre-identified artifact

A terminal pane already showing `docker logs -f m10-demo-weaviate`.


In [7]:
%%writefile demo/main_v4.py
# ── demo/main_v4.py ──────────────────────────────────────────────────────────────
# Beat 4: adds /healthz (liveness) and /readyz (readiness with two backend probes).
# All v3 code is retained; only the new endpoints are shown with full comments.

import re                                                           # Regex — used in extract_citations to find [N] citation markers
import time as _time                                                # Renamed to avoid shadowing the time module in lifespan
from contextlib import asynccontextmanager                          # Decorator for the FastAPI startup/shutdown lifespan handler
from typing import List                                             # Generic type for typed list annotations

from fastapi import FastAPI, Depends, Request                       # Core framework + DI helpers + request access
from fastapi.responses import JSONResponse                          # Used to set explicit HTTP status codes
from pydantic import BaseModel                                      # Base class for all request/response schema models
import spacy                                                        # NLP library — NER, POS tagging, dependency parsing
from neo4j import AsyncGraphDatabase, AsyncDriver                   # Async Neo4j driver — non-blocking Cypher execution
import weaviate                                                     # Weaviate vector-database client (sync v3 API)
from transformers import pipeline as hf_pipeline                    # Hugging Face inference pipeline — wraps flan-t5-base
import httpx                                                        # Async HTTP client used in /readyz probe

# ── Constants (same as v3) ────────────────────────────────────────────────────
NEO4J_URI      = "bolt://localhost:7688"                            # Bolt protocol URL for the demo Neo4j instance
NEO4J_USER     = "neo4j"                                            # Default Neo4j username
NEO4J_PASSWORD = "password"                                         # Demo password — replace with secrets manager in production
WEAVIATE_URL   = "http://localhost:8081"                            # HTTP URL for the demo Weaviate instance
RAG_CLASS      = "MovieReview"                                      # Weaviate class that holds the chunked review documents
TOP_K          = 4                                                  # Number of nearest-neighbour chunks to retrieve per query
GROUNDING_SENTINEL = "I cannot answer this question from the provided sources."  # Returned when the model cites nothing
PROMPT_TEMPLATE = (                                                 # Template injected with context chunks and the user question
    "You are a movie expert. Use ONLY the excerpts below to answer. "
    "Cite each excerpt you use with its number in square brackets.\n\n"
    "{context}\n\nQuestion: {question}\nAnswer:"
)

nlp = spacy.load("en_core_web_sm")                                  # Loaded once at module scope — shared across all requests

@asynccontextmanager
async def lifespan(app: FastAPI):                                   # Called once at startup (before yield) and shutdown (after yield)
    driver: AsyncDriver = AsyncGraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))  # Open async Neo4j connection pool
    app.state.neo4j_driver = driver                                 # Store on app.state so endpoints can retrieve it via Request
    wv_client = weaviate.Client(url=WEAVIATE_URL)                   # Create synchronous Weaviate client
    app.state.weaviate_client = wv_client                           # Store on app.state for endpoint access
    generator = hf_pipeline("text2text-generation", model="google/flan-t5-base")  # Load flan-t5-base into memory
    app.state.generator = generator                                 # Store pipeline on app.state for reuse across requests
    app.state.start_time = _time.time()                             # Record startup timestamp for /healthz uptime
    yield                                                           # Yield hands control to the running app; shutdown code runs after
    await driver.close()                                            # Close the Neo4j connection pool on shutdown

app = FastAPI(title="Movie Information Service — SI Demo", version="4.0.0", lifespan=lifespan)  # Register lifespan handler

# ── v1–v3 models and endpoints (abbreviated — same code as v3) ───────────────
class ExtractRequest(BaseModel):
    text: str                                                       # Raw text to run named-entity recognition on

class Entity(BaseModel):
    text: str; label: str; start: int; end: int                     # One recognised span: surface form, type, char offsets

class ExtractResponse(BaseModel):
    entities: List[Entity]; entity_count: int                       # All recognised entities + convenience count

@app.post("/extract", response_model=ExtractResponse, tags=["NLP"])
async def extract_entities(request: ExtractRequest) -> ExtractResponse:
    doc = nlp(request.text)                                         # Run the full spaCy pipeline: tokenise → POS → dep → NER
    entities = [Entity(text=e.text, label=e.label_, start=e.start_char, end=e.end_char) for e in doc.ents]  # Build typed entity list
    return ExtractResponse(entities=entities, entity_count=len(entities))

async def get_driver(request: Request) -> AsyncDriver:              # Dependency function — injected into kg_query via Depends()
    return request.app.state.neo4j_driver                           # Retrieve the driver stored during lifespan startup

class KGQueryRequest(BaseModel):
    cypher: str; params: dict = {}                                  # Cypher query string + optional parameter dict

@app.post("/kg/query", tags=["Knowledge Graph"])
async def kg_query(body: KGQueryRequest, driver: AsyncDriver = Depends(get_driver)) -> dict:
    async with driver.session() as session:                         # Open a new Neo4j session for this request
        result = await session.run(body.cypher, body.params)        # Execute the parameterised Cypher query
        records = [dict(r) async for r in result]                   # Async-iterate over result rows and convert to plain dicts
    return {"records": records, "count": len(records)}              # Return rows and total count

def assemble_prompt(chunks, question):                              # Builds the flan-t5 prompt from retrieved chunks
    context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks))  # Number each chunk for citation references
    return PROMPT_TEMPLATE.format(context=context, question=question)  # Inject context and question into the template

class CitedChunk(BaseModel):
    index: int; text: str                                           # Citation number and the original chunk text

class RAGRequest(BaseModel):
    question: str                                                   # Natural-language question from the user

class RAGResponse(BaseModel):
    answer: str; citations: List[CitedChunk]; grounded: bool        # Generated answer, source chunks used, grounding flag

def extract_citations(answer, chunks):                              # Parse [N] markers from the model output
    raw = re.findall(r"\[(\d+)\]", answer)                      # Find all [digit] patterns in the answer text
    unique = list(dict.fromkeys(int(i) for i in raw))               # Deduplicate while preserving order of first appearance
    return [CitedChunk(index=i, text=chunks[i-1]) for i in unique if 1 <= i <= len(chunks)]  # Map indices to chunk texts

@app.post("/rag/answer", response_model=RAGResponse, tags=["RAG"])
async def rag_answer(body: RAGRequest, request: Request) -> RAGResponse:
    wv = request.app.state.weaviate_client                          # Retrieve the Weaviate client stored at startup
    gen = request.app.state.generator                               # Retrieve the flan-t5 pipeline stored at startup
    result = wv.query.get(RAG_CLASS, ["chunk_text"]).with_near_text({"concepts": [body.question]}).with_limit(TOP_K).do()  # Vector search
    chunks = [h["chunk_text"] for h in result["data"]["Get"][RAG_CLASS]]  # Extract chunk text strings from Weaviate response
    prompt = assemble_prompt(chunks, body.question)                 # Build the numbered-context prompt
    raw_answer = gen(prompt, max_new_tokens=256, do_sample=False)[0]["generated_text"].strip()  # Run greedy generation
    citations = extract_citations(raw_answer, chunks)               # Parse [N] markers to identify which chunks were cited
    if not citations:                                               # No citations means the model did not ground its answer
        return RAGResponse(answer=GROUNDING_SENTINEL, citations=[], grounded=False)
    return RAGResponse(answer=raw_answer, citations=citations, grounded=True)

# ─────────────────────────────────────────────────────────────────────────────
# BEAT 4: HEALTH ENDPOINTS
# ─────────────────────────────────────────────────────────────────────────────

# GET /healthz — liveness probe
# Returns 200 as long as the process is running and the event loop is responsive.
# No backend checks — fast by design (K8s/Docker health check calls this frequently).
@app.get("/healthz", tags=["Ops"])
async def healthz(request: Request) -> dict:
    uptime = _time.time() - request.app.state.start_time            # Seconds since startup
    return {
        "status": "ok",                                             # Always "ok" when this endpoint is reachable
        "uptime_s": round(uptime, 2),                               # Useful for diagnosing recent restarts
    }

# GET /readyz — readiness probe
# Returns 200 only when BOTH backends are reachable.
# Returns 503 (Service Unavailable) if either backend fails its probe.
# Kubernetes uses 503 to remove the pod from the Service endpoint slice.
@app.get("/readyz", tags=["Ops"])
async def readyz(request: Request) -> JSONResponse:
    checks = {}                                                     # Collects pass/fail for each backend

    # ── Neo4j probe ──────────────────────────────────────────────────────────
    try:
        # Run a trivially cheap Cypher query — just returns the integer 1
        driver: AsyncDriver = request.app.state.neo4j_driver
        async with driver.session() as session:
            await session.run("RETURN 1")                           # If this succeeds, Neo4j is reachable
        checks["neo4j"] = "ok"
    except Exception as exc:
        checks["neo4j"] = f"error: {exc}"                           # Capture the error message for diagnostics

    # ── Weaviate probe ───────────────────────────────────────────────────────
    try:
        # Weaviate exposes a readiness endpoint at /v1/.well-known/ready
        async with httpx.AsyncClient() as client:
            wv_resp = await client.get(
                f"{WEAVIATE_URL}/v1/.well-known/ready",
                timeout=2.0,                                        # Fail fast — we do not want the probe to hang
            )
        # Weaviate returns 200 when it is ready, 503 when it is not
        checks["weaviate"] = "ok" if wv_resp.status_code == 200 else f"http {wv_resp.status_code}"
    except Exception as exc:
        checks["weaviate"] = f"error: {exc}"

    # ── Decision ─────────────────────────────────────────────────────────────
    all_ok = all(v == "ok" for v in checks.values())                # Both must pass
    http_status = 200 if all_ok else 503                            # 503 = tell LB to stop routing

    return JSONResponse(
        status_code=http_status,
        content={
            "status": "ready" if all_ok else "not_ready",           # Human-readable summary
            "checks": checks,                                       # Per-backend detail
        },
    )


Overwriting demo/main_v4.py


In [8]:
# ── Beat 4: Smoke Test — /healthz and /readyz ────────────────────────────────────

import subprocess, time, json, requests                             # Standard-library + requests for HTTP calls

DEMO_PORT = 8001                                                    # Same port as other beats — stop/start between them

server = subprocess.Popen(                                          # Launch v4 (health endpoints) as a background subprocess
    ["uvicorn", "demo.main_v4:app",                                 # Module path: demo/main_v4.py → app object
     "--port", str(DEMO_PORT),                                      # Bind to the demo port
     "--log-level", "warning"],                                     # Suppress INFO noise in notebook output
    stderr=subprocess.PIPE,                                         # Capture stderr to avoid cluttering output
)
print("Waiting for server + model load (up to 45 s) ...")          # flan-t5-base takes ~20-30 s to load
time.sleep(30)                                                      # Wait for uvicorn + model to initialise

try:                                                                # Guard against network errors if server did not start
    # ── /healthz ─────────────────────────────────────────────────────────────
    h = requests.get(f"http://localhost:{DEMO_PORT}/healthz", timeout=5)  # Liveness probe — always 200 if process is reachable
    print(f"/healthz  HTTP {h.status_code}")                        # Should be 200 when the server is running
    print(json.dumps(h.json(), indent=2))                           # Pretty-print uptime and status fields

    # ── /readyz ──────────────────────────────────────────────────────────────
    r = requests.get(f"http://localhost:{DEMO_PORT}/readyz", timeout=5)   # Readiness probe — 200 or 503 depending on backends
    print(f"\n/readyz  HTTP {r.status_code}")                      # 200 = both backends ready; 503 = at least one failed
    print(json.dumps(r.json(), indent=2))                           # Show per-backend check results (neo4j, weaviate)
    print()                                                         # Blank line for visual separation
    if r.status_code == 503:                                        # 503 means at least one backend is unreachable
        # Expected when demo containers are not running
        print("503 is expected if m10-demo-neo4j or m10-demo-weaviate are not up.")
        print("Start them and re-run to see the probe flip to 200.")

except requests.exceptions.RequestException as exc:                 # Catches connection refused, timeout, HTTP errors
    print(f"Request failed: {exc}")                                 # Surface the error without crashing the kernel

finally:                                                            # Always runs — cleans up even if an exception occurred
    server.terminate()                                              # Send SIGTERM to stop the uvicorn process
    server.wait()                                                   # Block until the process exits cleanly
    print("Server stopped.")                                        # Confirm cleanup so the port is free for the next beat


Waiting for server + model load (up to 45 s) ...


/tmp/ipykernel_23804/2757862256.py:7: ResourceWarning: unclosed file <_io.BufferedReader name=77>
  server = subprocess.Popen(                                          # Launch v4 (health endpoints) as a background subprocess


Request failed: HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /healthz (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ab9c18c16e0>: Failed to establish a new connection: [Errno 111] Connection refused'))
Server stopped.


---
## Beat 5 · CORS + Clickability from a Browser  *(5 min)*

### What is CORS and why it matters

When a browser JavaScript application on `http://localhost:3000` calls
`http://localhost:8000/extract`, the browser **blocks the request** by default
because the two ports are considered different origins.

The browser first sends an `OPTIONS` preflight request. Your API must respond with:

```
Access-Control-Allow-Origin: http://localhost:3000
Access-Control-Allow-Methods: GET, POST, OPTIONS
Access-Control-Allow-Headers: Content-Type
```

FastAPI's `CORSMiddleware` adds these headers automatically.

### Demo: before and after

**Before** (without middleware — browser console):
```
Access to fetch at 'http://localhost:8001/extract' from origin
'http://localhost:3001' has been blocked by CORS policy.
```

**After** (with middleware — network panel shows `200 OK`).

### Pre-identified artifact

A browser tab open on the Swagger UI with the network panel visible.


In [9]:
%%writefile demo/main_final.py
# ── demo/main_final.py ───────────────────────────────────────────────────────────
# Beat 5: adds CORSMiddleware.
# This is the complete, production-ready SI demo API.
# Lab assignment: replicate this pattern in your recipe-domain API.

import re, time as _time                                            # re for citation parsing; _time renamed to avoid shadowing
from contextlib import asynccontextmanager                          # Decorator for FastAPI startup/shutdown lifespan handler
from typing import List                                             # Generic type for typed list annotations

from fastapi import FastAPI, Depends, Request                       # Core framework + DI helpers + request access
from fastapi.responses import JSONResponse                          # Used to set explicit HTTP status codes
# CORSMiddleware: inspects Origin header and adds Access-Control-* response headers
from fastapi.middleware.cors import CORSMiddleware                  # Beat 5 addition — not present in v4
from pydantic import BaseModel                                      # Base class for all request/response schema models
import spacy                                                        # NLP library — NER, POS tagging, dependency parsing
from neo4j import AsyncGraphDatabase, AsyncDriver                   # Async Neo4j driver — non-blocking Cypher execution
import weaviate                                                     # Weaviate vector-database client (sync v3 API)
from transformers import pipeline as hf_pipeline                    # Hugging Face inference pipeline — wraps flan-t5-base
import httpx                                                        # Async HTTP client used in /readyz probe

# ── Constants ─────────────────────────────────────────────────────────────────
NEO4J_URI      = "bolt://localhost:7688"                            # Bolt protocol URL for the demo Neo4j instance
NEO4J_USER     = "neo4j"                                            # Default Neo4j username
NEO4J_PASSWORD = "password"                                         # Demo password — replace with secrets manager in production
WEAVIATE_URL   = "http://localhost:8081"                            # HTTP URL for the demo Weaviate instance
RAG_CLASS      = "MovieReview"                                      # Weaviate class that holds the chunked review documents
TOP_K          = 4                                                  # Number of nearest-neighbour chunks to retrieve per query
GROUNDING_SENTINEL = "I cannot answer this question from the provided sources."  # Returned when the model cites nothing
PROMPT_TEMPLATE = (                                                 # Template injected with context chunks and the user question
    "You are a movie expert. Use ONLY the excerpts below to answer. "
    "Cite each excerpt you use with its number in square brackets.\n\n"
    "{context}\n\nQuestion: {question}\nAnswer:"
)

nlp = spacy.load("en_core_web_sm")                                  # Loaded once at module scope

@asynccontextmanager
async def lifespan(app: FastAPI):                                   # Called once at startup (before yield) and shutdown (after yield)
    driver: AsyncDriver = AsyncGraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))  # Open async Neo4j connection pool
    app.state.neo4j_driver = driver                                 # Store driver on app.state for endpoint access
    app.state.weaviate_client = weaviate.Client(url=WEAVIATE_URL)   # Create and store Weaviate client
    app.state.generator = hf_pipeline("text2text-generation", model="google/flan-t5-base")  # Load flan-t5-base into memory
    app.state.start_time = _time.time()                             # Record startup timestamp for /healthz uptime
    yield                                                           # Yield hands control to the running app
    await driver.close()                                            # Close the Neo4j connection pool on shutdown

app = FastAPI(title="Movie Information Service — SI Demo", version="5.0.0", lifespan=lifespan)

# ── CORS Middleware ───────────────────────────────────────────────────────────
# add_middleware() must be called before any route definitions.
# allow_origins: list the exact frontend URLs that may call this API.
# For development: ["*"] allows all origins — never use "*" in production.
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",                                    # Lab Next.js dev server
        "http://localhost:3001",                                    # SI demo Next.js dev server
    ],
    allow_credentials=True,                                         # Allow cookies / Authorization headers
    allow_methods=["*"],                                            # Allow GET, POST, OPTIONS, etc.
    allow_headers=["*"],                                            # Allow Content-Type, Authorization, etc.
)

# ── All endpoints from v4 (unchanged) ────────────────────────────────────────
class ExtractRequest(BaseModel):
    text: str                                                       # Raw text to run named-entity recognition on

class Entity(BaseModel):
    text: str; label: str; start: int; end: int                     # One recognised span: surface form, type, char offsets

class ExtractResponse(BaseModel):
    entities: List[Entity]; entity_count: int                       # All recognised entities + convenience count

@app.post("/extract", response_model=ExtractResponse, tags=["NLP"])
async def extract_entities(request: ExtractRequest) -> ExtractResponse:
    doc = nlp(request.text)                                         # Run the full spaCy pipeline: tokenise → POS → dep → NER
    entities = [Entity(text=e.text, label=e.label_, start=e.start_char, end=e.end_char) for e in doc.ents]  # Build typed entity list
    return ExtractResponse(entities=entities, entity_count=len(entities))

async def get_driver(request: Request) -> AsyncDriver:              # Dependency function — injected via Depends()
    return request.app.state.neo4j_driver                           # Retrieve the driver stored during lifespan startup

class KGQueryRequest(BaseModel):
    cypher: str; params: dict = {}                                  # Cypher query string + optional parameter dict

@app.post("/kg/query", tags=["Knowledge Graph"])
async def kg_query(body: KGQueryRequest, driver: AsyncDriver = Depends(get_driver)) -> dict:
    async with driver.session() as session:                         # Open a new Neo4j session for this request
        result = await session.run(body.cypher, body.params)        # Execute the parameterised Cypher query
        records = [dict(r) async for r in result]                   # Async-iterate and convert rows to plain dicts
    return {"records": records, "count": len(records)}              # Return rows and total count

def assemble_prompt(chunks, question):                              # Builds the flan-t5 prompt from retrieved chunks
    context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks))  # Number each chunk for citation references
    return PROMPT_TEMPLATE.format(context=context, question=question)  # Inject context and question into the template

class CitedChunk(BaseModel):
    index: int; text: str                                           # Citation number and the original chunk text

class RAGRequest(BaseModel):
    question: str                                                   # Natural-language question from the user

class RAGResponse(BaseModel):
    answer: str; citations: List[CitedChunk]; grounded: bool        # Generated answer, source chunks, grounding flag

def extract_citations(answer, chunks):                              # Parse [N] markers from the model output
    raw = re.findall(r"\[(\d+)\]", answer)                      # Find all [digit] patterns in the answer text
    unique = list(dict.fromkeys(int(i) for i in raw))               # Deduplicate while preserving order of first appearance
    return [CitedChunk(index=i, text=chunks[i-1]) for i in unique if 1 <= i <= len(chunks)]  # Map indices to chunk texts

@app.post("/rag/answer", response_model=RAGResponse, tags=["RAG"])
async def rag_answer(body: RAGRequest, request: Request) -> RAGResponse:
    wv = request.app.state.weaviate_client                          # Retrieve the Weaviate client stored at startup
    gen = request.app.state.generator                               # Retrieve the flan-t5 pipeline stored at startup
    result = wv.query.get(RAG_CLASS, ["chunk_text"]).with_near_text({"concepts": [body.question]}).with_limit(TOP_K).do()  # Vector search
    chunks = [h["chunk_text"] for h in result["data"]["Get"][RAG_CLASS]]  # Extract chunk text strings from response
    prompt = assemble_prompt(chunks, body.question)                 # Build the numbered-context prompt
    raw_answer = gen(prompt, max_new_tokens=256, do_sample=False)[0]["generated_text"].strip()  # Run greedy generation
    citations = extract_citations(raw_answer, chunks)               # Parse [N] markers to identify cited chunks
    if not citations:                                               # No citations means the model did not ground its answer
        return RAGResponse(answer=GROUNDING_SENTINEL, citations=[], grounded=False)
    return RAGResponse(answer=raw_answer, citations=citations, grounded=True)

@app.get("/healthz", tags=["Ops"])
async def healthz(request: Request) -> dict:                        # Liveness probe — returns 200 as long as process is alive
    uptime = _time.time() - request.app.state.start_time            # Seconds elapsed since the app started
    return {"status": "ok", "uptime_s": round(uptime, 2)}          # "ok" is always returned when this endpoint is reachable

@app.get("/readyz", tags=["Ops"])
async def readyz(request: Request) -> JSONResponse:                 # Readiness probe — 200 only when both backends are reachable
    checks = {}                                                     # Collects per-backend pass/fail results
    try:
        async with request.app.state.neo4j_driver.session() as session:  # Open a temporary Neo4j session
            await session.run("RETURN 1")                           # Cheap query — verifies Neo4j connectivity
        checks["neo4j"] = "ok"
    except Exception as exc:
        checks["neo4j"] = f"error: {exc}"                           # Capture error detail for diagnostics
    try:
        async with httpx.AsyncClient() as client:
            wv_resp = await client.get(f"{WEAVIATE_URL}/v1/.well-known/ready", timeout=2.0)  # Weaviate readiness endpoint
        checks["weaviate"] = "ok" if wv_resp.status_code == 200 else f"http {wv_resp.status_code}"
    except Exception as exc:
        checks["weaviate"] = f"error: {exc}"
    all_ok = all(v == "ok" for v in checks.values())                # True only if every backend passed
    return JSONResponse(
        status_code=200 if all_ok else 503,                         # 503 tells the load balancer to stop routing traffic here
        content={"status": "ready" if all_ok else "not_ready", "checks": checks},  # Human-readable summary + per-backend detail
    )


Overwriting demo/main_final.py


In [10]:
# ── Beat 5: Observe CORS Headers ─────────────────────────────────────────────────
# We send an OPTIONS preflight and a POST from a simulated browser Origin header.
# The Access-Control-Allow-Origin header in the response proves CORS is working.

import subprocess, time, requests                                   # subprocess for server management, requests for HTTP calls

DEMO_PORT = 8001                                                    # Demo port — same as previous beats

server = subprocess.Popen(                                          # Launch main_final (v5 with CORSMiddleware) in background
    ["uvicorn", "demo.main_final:app",                              # Module: demo/main_final.py → app with CORS configured
     "--port", str(DEMO_PORT),                                      # Bind to the demo port
     "--log-level", "warning"],                                     # Suppress uvicorn INFO logs
    stderr=subprocess.PIPE,                                         # Capture stderr to keep output clean
)
print("Waiting for server + model load (up to 45 s) ...")          # flan-t5-base takes ~20-30 s to load
time.sleep(30)                                                      # Wait for uvicorn + model initialisation to complete

try:                                                                # Guard against network errors
    # Simulate a browser OPTIONS preflight — includes the Origin header the browser sends
    preflight = requests.options(                                    # OPTIONS verb — browser sends this before every cross-origin POST
        f"http://localhost:{DEMO_PORT}/extract",                    # The actual endpoint the browser wants to reach
        headers={
            "Origin": "http://localhost:3001",                      # Pretend we are the Next.js app
            "Access-Control-Request-Method": "POST",                # Asking permission to send a POST request
            "Access-Control-Request-Headers": "Content-Type",       # Declaring the headers the POST will include
        },
        timeout=5,                                                  # Fail fast if the server is not up
    )
    print(f"OPTIONS /extract  HTTP {preflight.status_code}")        # Should be 200 — preflight approved by CORSMiddleware
    # These are the CORS headers the browser checks
    for header in ["Access-Control-Allow-Origin",                   # Must match the request Origin for the browser to proceed
                   "Access-Control-Allow-Methods",                  # Lists which HTTP verbs are allowed
                   "Access-Control-Allow-Headers"]:                 # Lists which request headers are allowed
        val = preflight.headers.get(header, "(missing)")            # "(missing)" flags a misconfigured CORS setup
        print(f"  {header}: {val}")                                 # Print each CORS response header and its value

except requests.exceptions.RequestException as exc:                 # Catches connection refused, timeout, HTTP errors
    print(f"Request failed: {exc}")                                 # Surface the error without crashing the kernel

finally:                                                            # Always executes — guaranteed cleanup
    server.terminate()                                              # Send SIGTERM to stop the uvicorn process
    server.wait()                                                   # Block until the process exits cleanly
    print("\nServer stopped.")                                     # Confirm cleanup completed


/tmp/ipykernel_23804/338123557.py:9: ResourceWarning: unclosed file <_io.BufferedReader name=79>
  server = subprocess.Popen(                                          # Launch main_final (v5 with CORSMiddleware) in background


Waiting for server + model load (up to 45 s) ...
Request failed: HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /extract (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ab9c18c1f30>: Failed to establish a new connection: [Errno 111] Connection refused'))

Server stopped.


---
## Beat 6 · Next.js Page Scaffold for Python Engineers  *(15 min)*

### Key concepts for Python engineers

| Next.js concept | Python analogy |
|----------------|----------------|
| `pages/extract.tsx` → URL `/extract` | Flask `@app.route("/extract")` |
| `useState(initialValue)` | A mutable variable that triggers a re-render when changed |
| `fetch(url, { method: 'POST', body: JSON.stringify(data) })` | `requests.post(url, json=data)` |
| `NEXT_PUBLIC_API_URL` in `.env.local` | Environment variable loaded by `python-dotenv` |
| `npm run dev` | `uvicorn main:app --reload` |

### File-based routing

```
pages/
  index.tsx       →  http://localhost:3000/
  extract.tsx     →  http://localhost:3000/extract
  rag.tsx         →  http://localhost:3000/rag
```

No routing table needed — the filename **is** the route.

### The `NEXT_PUBLIC_` convention

- Variables prefixed `NEXT_PUBLIC_` are embedded in the client bundle at build time
- They are visible in the browser (do not store secrets in them)
- Variables without the prefix are **server-side only** (safe for secrets)

```bash
# .env.local
NEXT_PUBLIC_API_URL=http://localhost:8001   # Browser-side — must use NEXT_PUBLIC_
DB_PASSWORD=secret123                       # Server-side only — no NEXT_PUBLIC_
```

### Host vs. container URL

The browser runs on the **host** machine. `NEXT_PUBLIC_API_URL` must be the
**host-reachable** URL (`http://localhost:8001`), **not** the Docker-internal
service name (`http://api:8000`) — those are only reachable from inside the
Docker network.

### Pre-identified artifacts

- `pages/index.tsx` — pre-scaffolded (shows the entity picker UI)
- `.env.local` — pre-populated with `NEXT_PUBLIC_API_URL`


In [11]:
%%writefile demo/frontend/pages/extract.tsx
// ── demo/frontend/pages/extract.tsx ──────────────────────────────────────────────
// Beat 6: Next.js page for the /extract endpoint.
// File location determines the URL: pages/extract.tsx -> http://localhost:3001/extract

// React hooks — useState holds component state, triggering re-renders on change
import { useState } from "react";
// Next.js types — NextPage annotates that this export is a page component
import type { NextPage } from "next";

// ── TypeScript interface for the API response ─────────────────────────────────
// Mirrors the Python ExtractResponse Pydantic model exactly — same field names and types.
// This lets TypeScript catch mismatches between frontend and backend at compile time.
interface Entity {
  text: string;   // Surface form of the named entity
  label: string;  // spaCy entity type: PERSON, WORK_OF_ART, ORG, …
  start: number;  // Char offset start in original text
  end: number;    // Char offset end in original text
}

interface ExtractResponse {
  entities: Entity[];   // Array of recognised entity spans
  entity_count: number; // Convenience count field
}

// ── Page component ─────────────────────────────────────────────────────────────
// NextPage is a typed version of React.FC that Next.js recognises as a route handler
const ExtractPage: NextPage = () => {
  // inputText: the value bound to the textarea; setInputText updates it on each keystroke
  const [inputText, setInputText] = useState<string>("");

  // result: holds the parsed API response (null until the user submits)
  const [result, setResult] = useState<ExtractResponse | null>(null);

  // loading: tracks in-flight requests — disables the button and shows a spinner
  const [loading, setLoading] = useState<boolean>(false);

  // error: displays any network or API error to the user
  const [error, setError] = useState<string | null>(null);

  // ── handleSubmit: called when the user clicks "Extract Entities" ────────────
  const handleSubmit = async () => {
    setLoading(true);   // Disable button + show loading state
    setError(null);     // Clear any previous error
    setResult(null);    // Clear previous results

    // NEXT_PUBLIC_API_URL is read from .env.local at build time
    // Must be the host-reachable URL, not the Docker-internal service name
    const apiUrl = process.env.NEXT_PUBLIC_API_URL ?? "http://localhost:8001";

    try {
      // fetch() is the browser's built-in HTTP client (analogous to requests.post)
      const response = await fetch(`${apiUrl}/extract`, {
        method: "POST",                               // HTTP verb
        headers: { "Content-Type": "application/json" }, // Tell FastAPI to parse JSON
        body: JSON.stringify({ text: inputText }),    // Serialize request body
      });

      if (!response.ok) {
        // FastAPI returns 422 for validation errors — surface the detail to the user
        const err = await response.json();
        throw new Error(JSON.stringify(err.detail ?? err));
      }

      // Parse the JSON body and store it; TypeScript checks it matches ExtractResponse
      const data: ExtractResponse = await response.json();
      setResult(data);

    } catch (err: unknown) {
      // Catch both network errors and JSON parse errors
      setError(err instanceof Error ? err.message : String(err));
    } finally {
      setLoading(false);  // Always re-enable the button
    }
  };

  // ── Render ──────────────────────────────────────────────────────────────────
  return (
    <main style={{ fontFamily: "sans-serif", maxWidth: 720, margin: "2rem auto", padding: "0 1rem" }}>
      <h1>Named Entity Extraction — Movie Domain</h1>

      {/* Textarea bound to inputText state — updates on every keystroke */}
      <textarea
        rows={5}
        style={{ width: "100%", fontSize: "1rem", marginBottom: "0.5rem" }}
        placeholder="Paste a movie description, review, or plot summary here…"
        value={inputText}
        onChange={(e) => setInputText(e.target.value)}  // Sync textarea to state
      />

      {/* Submit button — disabled while a request is in flight */}
      <button
        onClick={handleSubmit}
        disabled={loading || inputText.trim().length === 0}
        style={{ padding: "0.5rem 1.5rem", fontSize: "1rem" }}
      >
        {loading ? "Extracting…" : "Extract Entities"}
      </button>

      {/* Error message — shown only when error state is set */}
      {error && (
        <p style={{ color: "crimson", marginTop: "1rem" }}>Error: {error}</p>
      )}

      {/* Results table — shown only when result state is set */}
      {result && (
        <>
          <p style={{ marginTop: "1rem" }}>
            Found <strong>{result.entity_count}</strong> entities.
          </p>
          <table style={{ width: "100%", borderCollapse: "collapse" }}>
            <thead>
              <tr style={{ background: "#f0f0f0" }}>
                <th style={{ textAlign: "left", padding: "0.4rem" }}>Text</th>
                <th style={{ textAlign: "left", padding: "0.4rem" }}>Label</th>
                <th style={{ textAlign: "left", padding: "0.4rem" }}>Start</th>
                <th style={{ textAlign: "left", padding: "0.4rem" }}>End</th>
              </tr>
            </thead>
            <tbody>
              {/* Map each Entity to a table row */}
              {result.entities.map((ent, idx) => (
                <tr key={idx} style={{ borderBottom: "1px solid #ddd" }}>
                  <td style={{ padding: "0.4rem" }}>{ent.text}</td>
                  <td style={{ padding: "0.4rem" }}>
                    <code>{ent.label}</code>
                  </td>
                  <td style={{ padding: "0.4rem" }}>{ent.start}</td>
                  <td style={{ padding: "0.4rem" }}>{ent.end}</td>
                </tr>
              ))}
            </tbody>
          </table>
        </>
      )}
    </main>
  );
};

// Default export required by Next.js file-based routing
export default ExtractPage;


Overwriting demo/frontend/pages/extract.tsx


In [12]:
%%writefile demo/frontend/.env.local
# ── demo/frontend/.env.local ─────────────────────────────────────────────────────
# Next.js reads this file automatically in development (npm run dev).
# NEXT_PUBLIC_ prefix: embedded in the browser bundle — safe for URLs, never for secrets.
# The URL below targets the SI demo API; change to http://localhost:8000 for your Lab API.
NEXT_PUBLIC_API_URL=http://localhost:8001


Overwriting demo/frontend/.env.local


In [13]:
%%writefile demo/frontend/pages/index.tsx
// ── demo/frontend/pages/index.tsx ────────────────────────────────────────────────
// Pre-scaffolded landing page — file maps to URL http://localhost:3001/
// Your assignment: this file is already provided; you author the fetch calls.

import type { NextPage } from "next";                              // NextPage: typed React component recognised by Next.js router
import Link from "next/link";                                      // Next.js router-aware anchor — enables client-side navigation

// ── Page component ────────────────────────────────────────────────────────────
// Arrow function returning JSX directly (parentheses instead of braces = implicit return)
const HomePage: NextPage = () => (
  // <main> is the semantic HTML landmark; inline style centres the content and limits max width
  <main style={{ fontFamily: "sans-serif", maxWidth: 720, margin: "2rem auto", padding: "0 1rem" }}>
    <h1>Movie Information Service — SI Demo</h1>   {/* Page heading — matches the FastAPI app title */}
    <p>Select a tool below:</p>                     {/* Prompt displayed above the navigation list */}

    {/* Link renders an anchor tag but uses client-side navigation */}
    <ul>
      {/* href="/extract" maps to pages/extract.tsx via Next.js file-based routing */}
      <li><Link href="/extract">Named Entity Extraction (/extract)</Link></li>
      {/* href="/rag" maps to pages/rag.tsx — RAG question-answering page */}
      <li><Link href="/rag">Movie Q&amp;A with RAG (/rag)</Link></li>
    </ul>

    <hr />  {/* Visual divider separating navigation links from the footer area */}
    <p style={{ color: "#888", fontSize: "0.85rem" }}>
      {/* NEXT_PUBLIC_API_URL is injected at build time from .env.local — safe in the browser bundle */}
      API docs: <a href={`${process.env.NEXT_PUBLIC_API_URL}/docs`} target="_blank" rel="noreferrer">
        {/* /docs is FastAPI's built-in Swagger UI, auto-generated from route decorators */}
        {process.env.NEXT_PUBLIC_API_URL}/docs
      </a>
    </p>
  </main>
);

export default HomePage;  // Required: Next.js imports this as the page's default export


Overwriting demo/frontend/pages/index.tsx


---
## Beat 7 · Backend Dockerfile  *(10 min)*

### Design goals

- **Single-stage** — Python images do not benefit from multi-stage builds the way
  compiled languages do (no separate build artifact to discard)
- **Layer cache** — copy `requirements.txt` and install dependencies **before**
  copying source code, so a code-only change does not re-run `pip install`
- **Non-root user** — drop privileges inside the container for security

### Layer cache diagram

```
FROM python:3.11-slim          ← cached on host after first pull
COPY requirements.txt .        ← rarely changes
RUN pip install ...            ← cached as long as requirements.txt is unchanged
COPY . .                       ← changes on every code edit  (cache busted here)
CMD ["uvicorn", ...]
```

### Pre-identified artifact

`.dockerignore` is already in the demo directory.


In [14]:
%%writefile demo/.dockerignore
# ── demo/.dockerignore ───────────────────────────────────────────────────────────
# Files and directories excluded from the Docker build context.
# Reducing the build context speeds up 'docker build' and avoids leaking secrets.

__pycache__                                                         # Python bytecode cache — not needed inside the container
*.pyc                                                               # Compiled Python files — regenerated at runtime
.env                                                                # Local env file — secrets must be injected at runtime, not baked in
.env.local                                                          # Same — never bake secrets into images
.git                                                                # Git history — not needed and can be large
.gitignore                                                          # Git metadata
node_modules                                                        # Node dependencies — the frontend has its own build stage
frontend/.next                                                      # Next.js build output — rebuilt inside the container
*.log                                                               # Log files


Overwriting demo/.dockerignore


In [15]:
%%writefile demo/Dockerfile.backend
# ── demo/Dockerfile.backend ──────────────────────────────────────────────────────
# Single-stage Python Dockerfile for the FastAPI service.
# Build: docker build -f Dockerfile.backend -t movie-api .
# Run:   docker run -p 8000:8000 movie-api

# Base image: official Python 3.11 slim variant (~130 MB vs ~900 MB for full Debian)
# Pinning to 3.11-slim ensures reproducible builds across environments
FROM python:3.11-slim

# Set the working directory inside the container
# All subsequent COPY, RUN, and CMD paths are relative to /app
WORKDIR /app

# ── Layer-cache optimisation ─────────────────────────────────────────────────
# Copy ONLY the dependency manifest first.
# If requirements.txt does not change, Docker reuses the cached pip layer
# on every subsequent build — saving 2-5 minutes per build cycle.
COPY requirements.txt ./

# Install Python dependencies with no cache to keep the layer small
# --no-cache-dir: do not store the pip download cache inside the image layer
RUN pip install --no-cache-dir -r requirements.txt

# Download the spaCy English model — baked into the image so the container
# does not need internet access at runtime
RUN python -m spacy download en_core_web_sm

# ── Source code ──────────────────────────────────────────────────────────────
# Copied after pip install so code changes do not invalidate the pip cache layer
COPY . .

# ── Security: drop to non-root user ─────────────────────────────────────────
# Running as root inside the container is a security risk if the container is
# ever compromised. The 'nobody' user has no home directory and no shell.
USER nobody

# ── Runtime ──────────────────────────────────────────────────────────────────
# EXPOSE documents the port; does not publish it (use -p at docker run time)
EXPOSE 8000

# CMD is the default command — overridable at runtime with 'docker run ... <cmd>'
# --host 0.0.0.0: bind to all interfaces inside the container (required for Docker)
# --workers 1: single worker for demo; increase for production
CMD ["uvicorn", "main_final:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]


Overwriting demo/Dockerfile.backend


In [16]:
%%writefile demo/requirements.txt
# ── demo/requirements.txt ────────────────────────────────────────────────────────
# Pin major versions to avoid unexpected breaking changes.
# Run: pip install -r requirements.txt

fastapi==0.111.0                                                    # Web framework — routing, DI, OpenAPI
uvicorn[standard]==0.29.0                                           # ASGI server — websockets + httptools for performance
pydantic==2.7.1                                                     # Request/response validation and serialisation
neo4j==5.20.0                                                       # Official Neo4j async Python driver
spacy==3.8.3                                                        # NLP pipeline — NER, POS, dependency parsing
weaviate-client==3.26.2                                             # Weaviate v3 Python client (sync)
sentence-transformers==2.7.0                                        # Sentence embeddings — used by Weaviate at index time
transformers==4.40.1                                                # Hugging Face transformers — flan-t5-base generation
torch==2.3.0                                                        # PyTorch — required by transformers and sentence-transformers
httpx==0.27.0                                                       # Async HTTP client — used in /readyz Weaviate probe

Overwriting demo/requirements.txt


---
## Beat 8 · Frontend Dockerfile — Multi-Stage Node  *(10 min)*

### Why multi-stage?

Node.js builds produce a large `node_modules` directory (~700 MB) that is only
needed during compilation. The **runner** stage copies only the compiled output,
producing a final image of ~250 MB.

```
Stage 1: builder (~700 MB)       Stage 2: runner (~250 MB)
┌─────────────────────────┐      ┌──────────────────────────────┐
│ node:20-alpine           │  ──► │ node:20-alpine               │
│ npm ci                   │      │ COPY --from=builder .next ... │
│ npm run build            │      │ CMD ["node", "server.js"]    │
│  ↓ .next/ standalone     │      └──────────────────────────────┘
└─────────────────────────┘
```

### Pre-identified artifact

A slide with the side-by-side image-size comparison (builder ~700 MB; runner ~250 MB).


In [17]:
%%writefile demo/Dockerfile.frontend
# ── demo/Dockerfile.frontend ─────────────────────────────────────────────────────
# Two-stage Dockerfile for the Next.js frontend.
# Stage 1 (builder): installs all Node deps and compiles the app
# Stage 2 (runner):  copies only the compiled output — ~250 MB vs ~700 MB
#
# Build: docker build -f Dockerfile.frontend -t movie-web .
# Run:   docker run -p 3000:3000 -e NEXT_PUBLIC_API_URL=http://localhost:8000 movie-web

# ── Stage 1: builder ─────────────────────────────────────────────────────────
# node:20-alpine: Alpine Linux base keeps the image small
# AS builder: names this stage so stage 2 can reference it with --from=builder
FROM node:20-alpine AS builder

# Working directory for all build steps
WORKDIR /app

# Copy package manifests first (same layer-cache trick as the backend Dockerfile)
# package.json: declares dependencies and scripts
# package-lock.json: pins exact versions for reproducible installs
COPY package.json package-lock.json ./

# npm ci: clean install — faster than npm install; respects lockfile exactly
# --omit=dev: skip devDependencies (linters, type checkers not needed at build time)
RUN npm ci --omit=dev

# Copy source files after npm ci so a code change does not bust the npm cache layer
COPY . .

# next.config.js must set output: "standalone" for this Dockerfile to work
# Standalone output bundles only the files needed to run the app (no node_modules)
RUN npm run build

# ── Stage 2: runner ──────────────────────────────────────────────────────────
# Fresh node:20-alpine — no build tools, no package.json, no node_modules
FROM node:20-alpine AS runner

WORKDIR /app

# Security: create and use a non-root user for the production container
RUN addgroup --system --gid 1001 nodejs &&     adduser  --system --uid  1001 nextjs

# Copy the compiled standalone server from the builder stage
# --from=builder references the named build stage above
COPY --from=builder --chown=nextjs:nodejs /app/.next/standalone ./

# Copy static assets (CSS, images, fonts) — served directly without Node.js
COPY --from=builder --chown=nextjs:nodejs /app/.next/static ./.next/static

# Copy the public/ directory (favicons, og images, robots.txt, etc.)
COPY --from=builder --chown=nextjs:nodejs /app/public ./public

# Switch to the non-root user for runtime
USER nextjs

# Document the port the app listens on
EXPOSE 3000

# Set PORT and HOSTNAME explicitly — Next.js standalone respects these env vars
ENV PORT=3000
ENV HOSTNAME=0.0.0.0

# server.js is generated by 'next build' when output: "standalone" is set
CMD ["node", "server.js"]


Overwriting demo/Dockerfile.frontend


---
## Beat 9 · Lab-level Compose — `api` + `web`  *(5 min)*

### What the Compose file does

- Defines two services: `api` (FastAPI) and `web` (Next.js)
- Both services share a custom network — they address each other by service name
- `web` depends on `api` being healthy before it starts

### The host-vs-container URL problem (revisited)

The Next.js server-side code inside the container can reach FastAPI at
`http://api:8000` (Docker-internal DNS). But the browser on your host machine
cannot — it needs `http://localhost:8000`.

This is why `NEXT_PUBLIC_API_URL` uses `localhost` — it runs in the browser
on the host, not inside Docker.

### Pre-identified artifact

The Compose file printed on Slide 10.


In [18]:
%%writefile demo/docker-compose.yml
# ── demo/docker-compose.yml ──────────────────────────────────────────────────────
# Lab-level Compose: two services — api (FastAPI) and web (Next.js).
# Run: docker compose up -d
# Stop: docker compose down

services:

  # ── FastAPI backend ──────────────────────────────────────────────────────────
  api:
    build:
      context: .                                                    # Build context is the demo/ directory
      dockerfile: Dockerfile.backend                                # Use the single-stage Python Dockerfile
    image: movie-api:local                                          # Tag the built image for easy reference
    ports:
      - "8001:8000"                                                 # host:container — demo uses 8001 to avoid Lab collision
    environment:
      # These override the hard-coded constants in main_final.py
      # In production, inject these from a secrets manager, not plain env vars
      - NEO4J_URI=bolt://host.docker.internal:7688                  # Reach host's Neo4j from inside Docker
      - NEO4J_USER=neo4j
      - NEO4J_PASSWORD=password
      - WEAVIATE_URL=http://host.docker.internal:8081               # Reach host's Weaviate
    healthcheck:
      # Docker uses this to determine when the service is ready
      # curl -f returns non-zero on HTTP error codes, triggering an unhealthy state
      test: ["CMD", "curl", "-f", "http://localhost:8000/readyz"]
      interval: 10s                                                 # Check every 10 seconds
      timeout: 5s                                                   # Fail the check if it takes longer than 5 s
      retries: 3                                                    # Mark unhealthy after 3 consecutive failures
      start_period: 40s                                             # Grace period — model loading takes time

  # ── Next.js frontend ─────────────────────────────────────────────────────────
  web:
    build:
      context: ./frontend                                           # Build context is the frontend sub-directory
      dockerfile: ../Dockerfile.frontend                            # Dockerfile is one level up
    image: movie-web:local
    ports:
      - "3001:3000"                                                 # Demo uses 3001 to avoid Lab's port 3000
    environment:
      # NEXT_PUBLIC_API_URL must be the host-reachable URL because it runs in the browser
      # NOT http://api:8000 — that is internal to Docker and unreachable by the browser
      - NEXT_PUBLIC_API_URL=http://localhost:8001
    depends_on:
      api:
        condition: service_healthy                                  # Wait until api passes its healthcheck

# Shared network — services communicate using service names as hostnames
networks:
  default:
    name: m10-demo-net


Overwriting demo/docker-compose.yml


In [19]:
# ── Beat 9: Bring Up the Full Stack ──────────────────────────────────────────────
# Run 'docker compose up -d' inside the demo directory.
# The -d flag detaches — Compose runs in the background.

import subprocess                                                   # Used to run the docker compose command from Python

result = subprocess.run(                                            # Blocking call — waits for docker compose to finish starting
    ["docker", "compose", "up", "--build", "-d"],                   # --build forces image rebuild
    cwd="demo",                                                     # Run from the demo/ directory where docker-compose.yml lives
    capture_output=True,                                            # Capture stdout and stderr instead of printing to terminal
    text=True,                                                      # Decode bytes to str so we can print readable output
)

print(result.stdout)                                                # Print Compose's progress output
if result.stderr:                                                   # stderr may contain warnings even on a successful run
    print(result.stderr)                                            # Print any warnings or errors

if result.returncode == 0:                                          # Exit code 0 means all services started successfully
    print("Stack is up. Open http://localhost:3001 in your browser.")   # Frontend URL mapped by docker-compose.yml
    print("API Swagger UI: http://localhost:8001/docs")             # FastAPI interactive docs — auto-generated from routes
else:                                                               # Non-zero exit code means at least one service failed to start
    print(f"docker compose up failed (exit {result.returncode}). Check the output above.")


#0 building with "default" instance using docker driver

#1 [api internal] load build definition from Dockerfile.backend
#1 DONE 0.1s

#1 [api internal] load build definition from Dockerfile.backend
#1 transferring dockerfile:
#1 transferring dockerfile: 2.72kB 0.7s done
#1 DONE 2.6s

#2 [api internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 7.3s

#3 [api internal] load .dockerignore
#3 transferring context:
#3 transferring context: 1.42kB 0.0s done
#3 DONE 0.6s

#4 [api internal] load build context
#4 ...

#5 [api 1/6] FROM docker.io/library/python:3.11-slim@sha256:ae52c5bef62a6bdd42cd1e8dffef86b9cd284bde9427da79839de7a4b983e7ca
#5 resolve docker.io/library/python:3.11-slim@sha256:ae52c5bef62a6bdd42cd1e8dffef86b9cd284bde9427da79839de7a4b983e7ca 0.6s done
#5 DONE 0.6s

#4 [api internal] load build context
#4 transferring context: 44.53kB 3.1s
#4 transferring context: 186.90kB 8.1s
#4 transferring context: 477.40kB 13.2s
#4 transferring context: 890.35kB 17.1s done


---
## Beat 10 · Pivot to the Assignment  *(8 min)*

### Your domain is recipes

Everything you just watched in the movie domain maps directly to your recipe domain.
The Week 9B Neo4j graph and the Module 8 RAG corpus are pre-loaded in your Lab template.

| SI Demo (movies) | Your Assignment (recipes) |
|---|---|
| `ExtractRequest.text` → movie synopsis | `ExtractRequest.text` → recipe description |
| `Entity.label` → PERSON, WORK_OF_ART | `Entity.label` → INGREDIENT, DISH, TECHNIQUE |
| `(:Movie)-[:DIRECTED_BY]->(:Director)` | `(:Recipe)-[:USES]->(:Ingredient)` |
| `MovieReview` Weaviate class | `RecipeReview` Weaviate class |
| `NEXT_PUBLIC_API_URL=http://localhost:8001` | `NEXT_PUBLIC_API_URL=http://localhost:8000` |

---

### Decision points you will hit  *(read before you code)*

**A · Where to load the generator model**
Load it in `lifespan`, not in the endpoint function. The trap is real — the endpoint
will appear to work in testing but will be 20-30 s slow under load.

**B · How to surface `UnsupportedQueryError` as 422**
Raise `HTTPException(status_code=422, detail="...")` inside the route handler.
FastAPI converts it to a JSON `{"detail": "..."}` response automatically.

**C · How to enforce RAG grounding**
If `extract_citations(answer, chunks)` returns an empty list, return the grounding
sentinel. Do not return `raw_answer` when `citations` is empty.

**D · Which Next.js env var convention to use**
Always prefix browser-visible variables with `NEXT_PUBLIC_`. Variables without the
prefix are server-side only and will appear as `undefined` in the browser.

**E · Host-vs-container URL difference**
`NEXT_PUBLIC_API_URL` must be `http://localhost:8000` (host-reachable), not
`http://api:8000` (Docker-internal). The browser runs on the host, not inside Docker.

---

Open the Lab Guide now. Good luck.


In [1]:
# ── Teardown · Shutdown All Services ─────────────────────────────────────────────
# ALL LINES ARE COMMENTED OUT — uncomment and run this cell only when you are
# ready to tear everything down.

import subprocess                                                  # Shell commands for container and compose teardown

# ── 1. Terminate any lingering uvicorn server subprocess ──────────────────────
# If a 'server' variable is still alive from the last smoke-test cell, stop it.
try:
    server.terminate()                                            # Send SIGTERM to the uvicorn process
    server.wait()                                                 # Block until the process exits cleanly
    print("Uvicorn server stopped.")
except NameError:
    print("No active 'server' subprocess found — nothing to stop.")

# ── 2. Bring down the Docker Compose stack (api + web) ───────────────────────
# Stops and removes the 'api' (FastAPI) and 'web' (Next.js) containers defined
# in demo/docker-compose.yml. Volumes and built images are preserved.
result_compose = subprocess.run(
    ["docker", "compose", "down"],                               # Stop + remove containers and networks
    cwd="demo",                                                  # Run from demo/ where docker-compose.yml lives
    capture_output=True,
    text=True,
)
print(result_compose.stdout)
if result_compose.returncode == 0:
    print("Compose stack (api + web) stopped.")
else:
    print(f"docker compose down failed (exit {result_compose.returncode}): {result_compose.stderr.strip()}")

# ── 3. Stop the Neo4j knowledge-graph container ───────────────────────────────
# m10-demo-neo4j: host port 7688 → container port 7687
result_neo4j = subprocess.run(
    ["docker", "stop", "m10-demo-neo4j"],                        # Graceful SIGTERM → SIGKILL after 10 s
    capture_output=True, text=True,
)
if result_neo4j.returncode == 0:
    print("m10-demo-neo4j stopped.")
else:
    print(f"Could not stop m10-demo-neo4j: {result_neo4j.stderr.strip()}")

# ── 4. Stop the Weaviate vector-store container ───────────────────────────────
# m10-demo-weaviate: host port 8081 → container port 8080
result_weaviate = subprocess.run(
    ["docker", "stop", "m10-demo-weaviate"],                     # Graceful SIGTERM → SIGKILL after 10 s
    capture_output=True, text=True,
)
if result_weaviate.returncode == 0:
    print("m10-demo-weaviate stopped.")
else:
    print(f"Could not stop m10-demo-weaviate: {result_weaviate.stderr.strip()}")

# ── 5. Confirm no demo containers are still running ───────────────────────────
result_ps = subprocess.run(
    ["docker", "ps", "--filter", "name=m10-demo", "--filter", "name=movie-",
     "--format", "table {{.Names}}\t{{.Status}}\t{{.Ports}}"],
    capture_output=True, text=True,
)
print("\nRemaining demo containers:")
print(result_ps.stdout.strip() or "(none — teardown complete)")


No active 'server' subprocess found — nothing to stop.

Compose stack (api + web) stopped.
Could not stop m10-demo-neo4j: Error response from daemon: No such container: m10-demo-neo4j
Could not stop m10-demo-weaviate: Error response from daemon: No such container: m10-demo-weaviate

Remaining demo containers:
NAMES     STATUS    PORTS
